In [2]:
from pyspark.sql.functions import col, to_timestamp, when, regexp_replace
from pyspark.sql import SparkSession
import pandas as pd
import time
import warnings

warnings.filterwarnings('ignore')

In [3]:
start_time = time.time()
df = pd.read_csv('../Data/Crime_Records.csv')
crime_records = df[['Incident ID', 'Incident Description', 'Incident Datetime', 'Incident Day of Week', 'Incident Category', 'Incident Subcategory', 'Report Datetime', 'Report Type Code','Report Type Description', 'Police District', 'Latitude', 'Longitude', 'Resolution']]
   
# some transformations
crime_records['Incident Datetime'] = pd.to_datetime(crime_records['Incident Datetime'])
crime_records['Report Datetime'] = pd.to_datetime(crime_records['Report Datetime'])
crime_records['Incident Category'] = crime_records['Incident Category'].apply(str)
crime_records['Incident Subcategory'] = crime_records['Incident Subcategory'].apply(str)
crime_records['Incident Description'] = crime_records['Incident Description'].apply(lambda d: d.replace(',', '-') )
crime_records['Incident Category'] = crime_records['Incident Category'].apply(lambda d: d.replace(',', '-') )
crime_records['Incident Subcategory'] = crime_records['Incident Subcategory'].apply(lambda d: d.replace(',', '-') )
crime_records['Incident Category'].fillna('', inplace=True)
crime_records['Incident Subcategory'].fillna('', inplace=True)
crime_records.drop_duplicates(inplace=True)
crime_records.loc[crime_records['Incident Category']=='nan', ['Incident Subcategory', 'Incident Category']] = ''
print("--- %s seconds ---" % (time.time() - start_time))
crime_records.head(10)

--- 75.66992139816284 seconds ---


,Incident ID,Incident Description,Incident Datetime,Incident Day of Week,Incident Category,Incident Subcategory,Report Datetime,Report Type Code,Report Type Description,Police District,Latitude,Longitude,Resolution
0,1253736,Vehicle- Recovered- Auto,2023-03-13 23:41:00,Monday,Recovered Vehicle,Recovered Vehicle,2023-03-13 23:41:00,VS,Vehicle Supplement,Out of SF,NaN,NaN,Open or Active
1,1253795,Theft- Other Property- >$950,2023-03-01 05:02:00,Wednesday,Larceny Theft,Larceny Theft - Other,2023-03-11 15:40:00,II,Coplogic Initial,Mission,NaN,NaN,Open or Active
2,1253571,Vehicle- Recovered- Auto,2023-03-13 13:16:00,Monday,Recovered Vehicle,Recovered Vehicle,2023-03-13 13:17:00,VS,Vehicle Supplement,Out of SF,NaN,NaN,Open or Active
3,1253551,Vehicle- Recovered- Auto,2023-03-13 10:59:00,Monday,Recovered Vehicle,Recovered Vehicle,2023-03-13 11:00:00,VS,Vehicle Supplement,Out of SF,NaN,NaN,Open or Active
4,1254024,Vehicle- Recovered- Auto,2023-03-14 18:44:00,Tuesday,Recovered Vehicle,Recovered Vehicle,2023-03-14 18:45:00,VS,Vehicle Supplement,Out of SF,NaN,NaN,Open or Active
5,1253786,Theft- Other Property- $50-$200,2023-02-15 03:00:00,Wednesday,Larceny Theft,Larceny Theft - Other,2023-03-11 16:55:00,II,Coplogic Initial,Mission,NaN,NaN,Open or Active
6,1253816,Theft- From Locked Vehicle- >$950,2023-03-11 12:30:00,Saturday,Larceny Theft,Larceny - From Vehicle,2023-03-12 16:15:00,II,Coplogic Initial,Central,NaN,NaN,Open or Active
7,1254195,Theft- From Locked Vehicle- >$950,2023-03-13 11:26:00,Monday,Larceny Theft,Larceny - From Vehicle,2023-03-13 13:37:00,II,Coplogic Initial,Central,NaN,NaN,Open or Active
8,1254206,Theft- From Locked Vehicle- >$950,2023-03-11 15:00:00,Saturday,Larceny Theft,Larceny - From Vehicle,2023-03-13 08:29:00,IS,Coplogic Supplement,Central,NaN,NaN,Open or Active
9,1254318,Battery,2023-03-11 14:00:00,Saturday,Assault,Simple Assault,2023-03-15 11:21:00,II,Initial,Park,37.772895,-122.454285,Open or Active


In [8]:
spark = SparkSession.builder.appName('pyspark_tut').getOrCreate()

In [9]:
spark

In [10]:
start_time = time.time()
df_pyspark = spark.read.csv('../Data/Crime_Records.csv', header=True, inferSchema=True)
df_pyspark = df_pyspark.select('Incident ID', 'Incident Description', 'Incident Datetime', 'Incident Day of Week', 'Incident Category', 'Incident Subcategory', 'Report Datetime', 'Report Type Code', 'Report Type Description', 'Police District', 'Latitude', 'Longitude', 'Resolution')

# Convert datetime columns
df_pyspark = df_pyspark.withColumn("Incident Datetime", to_timestamp(col("Incident Datetime")))
df_pyspark = df_pyspark.withColumn("Report Datetime", to_timestamp(col("Report Datetime")))

# Ensure the categories are treated as string and handle commas in descriptions
df_pyspark = df_pyspark.withColumn("Incident Category", col("Incident Category").cast("string"))
df_pyspark = df_pyspark.withColumn("Incident Subcategory", col("Incident Subcategory").cast("string"))
df_pyspark = df_pyspark.withColumn("Incident Description", regexp_replace(col("Incident Description"), ",", "-"))
df_pyspark = df_pyspark.withColumn("Incident Category", regexp_replace(col("Incident Category"), ",", "-"))
df_pyspark = df_pyspark.withColumn("Incident Subcategory", regexp_replace(col("Incident Subcategory"), ",", "-"))

# Handle null values in categories and subcategories
df_pyspark = df_pyspark.fillna({'Incident Category': '', 'Incident Subcategory': ''})

# Drop duplicates
df_pyspark = df_pyspark.dropDuplicates()

# Replace 'nan' in 'Incident Category' with empty string and apply the same to 'Incident Subcategory'
df_pyspark = df_pyspark.withColumn(
    "Incident Category",
    when(col("Incident Category") == "nan", '').otherwise(col("Incident Category"))
)
df_pyspark = df_pyspark.withColumn(
    "Incident Subcategory",
    when(col("Incident Category") == '', '').otherwise(col("Incident Subcategory"))
)

print("--- %s seconds ---" % (time.time() - start_time))

--- 0.7365376949310303 seconds ---


In [11]:
df_pyspark.show(10)

+-----------+--------------------+-----------------+--------------------+------------------+--------------------+---------------+----------------+-----------------------+---------------+------------------+-------------------+--------------+
|Incident ID|Incident Description|Incident Datetime|Incident Day of Week| Incident Category|Incident Subcategory|Report Datetime|Report Type Code|Report Type Description|Police District|          Latitude|          Longitude|    Resolution|
+-----------+--------------------+-----------------+--------------------+------------------+--------------------+---------------+----------------+-----------------------+---------------+------------------+-------------------+--------------+
|    1049809|   Terrorist Threats|             null|              Monday|Disorderly Conduct|        Intimidation|           null|              II|                Initial|     Tenderloin| 37.78321431177312|-122.41076482950653|Open or Active|
|    1068216|       Lost Property|  